In [210]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns


from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.preprocessing import LabelEncoder,StandardScaler

import warnings
warnings.filterwarnings("ignore")



In [211]:
df1 = pd.read_csv(r"C:\Users\kpasw\OneDrive\Desktop\CP 2\5g_network_data.csv")
df1

,Location,Signal Strength (dBm),Download Speed (Mbps),Upload Speed (Mbps),Latency (ms),Jitter (ms),Network Type,Device Model,Carrier,Band,Battery Level (%),Temperature (°C),Connected Duration (min),Handover Count,Data Usage (MB),Video Streaming Quality,VoNR Enabled,Network Congestion Level,Ping to Google (ms),Dropped Connection
0,San Francisco,-108.6,714.94,60.41,10.0,4.09,5G NSA,iPhone 14,AT&T,n78,99,35.5,14,1,97.40,4,False,High,27.9,True
1,San Francisco,-71.5,686.69,148.70,12.3,1.50,4G,Pixel 7,AT&T,n260,67,22.0,51,4,143.23,3,True,Medium,22.2,False
2,Chennai,-76.1,428.14,74.00,14.0,0.10,5G SA,Galaxy S23,BSNL,n78,26,40.8,48,3,425.53,2,True,Low,36.6,False
3,New York,-73.3,208.56,68.59,12.2,4.94,4G,Pixel 7,T-Mobile,n41,25,39.3,48,0,128.87,4,False,High,87.5,False
4,Kolkata,-93.2,409.85,137.23,6.3,2.94,5G NSA,Galaxy S23,BSNL,n28,51,22.7,54,4,156.91,1,True,Medium,32.5,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20562,Tokyo,-94.2,770.26,57.62,7.7,0.58,5G SA,GT 7,Verizon,n260,71,44.0,24,2,379.00,4,False,High,75.8,True
20563,Berlin,-81.1,984.29,124.55,6.4,1.74,4G,Nord 4,T-Mobile,n78,58,21.3,11,1,196.33,4,True,Medium,72.7,True
20564,Delhi,-76.6,480.16,102.38,13.4,5.00,5G NSA,Pixel 7,Jio,n78,27,35.8,55,0,81.66,3,True,Low,15.2,True
20565,Kolkata,-62.2,851.07,78.59,8.0,2.16,5G NSA,Pixel 7,Jio,n78,31,24.4,55,1,38.16,3,True,Medium,64.2,True


In [212]:
drop_cols = [
    "Device Model",
    "Temperature (°C)",
    "Battery Level (%)",
    "Connected Duration (min)",
    "Data Usage (MB)",
    "Ping to Google (ms)",
    "Handover Count",
    "Carrier",
    "Band",
    "Location",
    "Network Type"
]

df = df1.drop(columns=drop_cols, errors="ignore")

In [213]:
df

,Signal Strength (dBm),Download Speed (Mbps),Upload Speed (Mbps),Latency (ms),Jitter (ms),Video Streaming Quality,VoNR Enabled,Network Congestion Level,Dropped Connection
0,-108.6,714.94,60.41,10.0,4.09,4,False,High,True
1,-71.5,686.69,148.70,12.3,1.50,3,True,Medium,False
2,-76.1,428.14,74.00,14.0,0.10,2,True,Low,False
3,-73.3,208.56,68.59,12.2,4.94,4,False,High,False
4,-93.2,409.85,137.23,6.3,2.94,1,True,Medium,True
...,...,...,...,...,...,...,...,...,...
20562,-94.2,770.26,57.62,7.7,0.58,4,False,High,True
20563,-81.1,984.29,124.55,6.4,1.74,4,True,Medium,True
20564,-76.6,480.16,102.38,13.4,5.00,3,True,Low,True
20565,-62.2,851.07,78.59,8.0,2.16,3,True,Medium,True


In [214]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20567 entries, 0 to 20566
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Signal Strength (dBm)     20567 non-null  float64
 1   Download Speed (Mbps)     20329 non-null  float64
 2   Upload Speed (Mbps)       20567 non-null  float64
 3   Latency (ms)              20567 non-null  float64
 4   Jitter (ms)               20567 non-null  float64
 5   Video Streaming Quality   20567 non-null  int64  
 6   VoNR Enabled              20450 non-null  object 
 7   Network Congestion Level  20280 non-null  object 
 8   Dropped Connection        20567 non-null  bool   
dtypes: bool(1), float64(5), int64(1), object(2)
memory usage: 1.3+ MB


In [215]:
df.isna().sum()

Signal Strength (dBm)         0
Download Speed (Mbps)       238
Upload Speed (Mbps)           0
Latency (ms)                  0
Jitter (ms)                   0
Video Streaming Quality       0
VoNR Enabled                117
Network Congestion Level    287
Dropped Connection            0
dtype: int64

In [216]:
# df['Download Speed (Mbps)'] = df['Download Speed (Mbps)'].fillna(df['col'].median())
# df['VoNR Enabled'] = df['col'].fillna(df['col'].mode()[0])
df = df.dropna()


In [217]:

congestion_map = {"Low":1, "Medium":2, "High":3}
df["Congestion_Num"] = df["Network Congestion Level"].map(congestion_map)

df["Quality_Score"] = (
    (df["Download Speed (Mbps)"] * 0.5)
    - (df["Latency (ms)"] * 0.2)
    - (df["Jitter (ms)"] * 0.1)
    - (df["Congestion_Num"] * 5)
    - (df["Dropped Connection"].astype(int) * 20)
)

df["Video Streaming Quality"] = df["Video Streaming Quality"].astype(int)

df["Video Streaming Quality"] = pd.qcut(
    df["Quality_Score"],
    q=5,
    labels=[1,2,3,4,5]
).astype(int)

df.drop(["Quality_Score","Congestion_Num"], axis=1, inplace=True)


In [218]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19942 entries, 0 to 20566
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Signal Strength (dBm)     19942 non-null  float64
 1   Download Speed (Mbps)     19942 non-null  float64
 2   Upload Speed (Mbps)       19942 non-null  float64
 3   Latency (ms)              19942 non-null  float64
 4   Jitter (ms)               19942 non-null  float64
 5   Video Streaming Quality   19942 non-null  int32  
 6   VoNR Enabled              19942 non-null  object 
 7   Network Congestion Level  19942 non-null  object 
 8   Dropped Connection        19942 non-null  bool   
dtypes: bool(1), float64(5), int32(1), object(2)
memory usage: 1.3+ MB


In [219]:
le = LabelEncoder()
df['VoNR Enabled'] = le.fit_transform(df['VoNR Enabled'])
df['Network Congestion Level'] = le.fit_transform(df['Network Congestion Level'])
df['Dropped Connection'] = le.fit_transform(df['Dropped Connection'])


In [220]:
# x=df.drop('Video Streaming Quality',axis=1)
# y=["Video Streaming Quality"]

In [221]:
features = [
    "Signal Strength (dBm)",
    "Download Speed (Mbps)",
    "Latency (ms)",
    "Jitter (ms)",
    "VoNR Enabled",
    "Dropped Connection"
]

x=df[features]
y=df["Video Streaming Quality"]


In [222]:
# x=pd.get_dummies(x, drop_first=True)

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [223]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


In [224]:
x_train.shape

(15953, 6)

In [225]:
x_test.shape

(3989, 6)

In [226]:
y_train.shape

(15953,)

In [227]:
y_test.shape

(3989,)

In [228]:
rf = RandomForestClassifier(random_state=42)


In [229]:
rf=rf.fit(x_train, y_train)


In [230]:
y_pred = rf.predict(x_test)


In [231]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.966658310353472
              precision    recall  f1-score   support

           1       0.97      0.98      0.98       798
           2       0.97      0.96      0.96       797
           3       0.96      0.96      0.96       798
           4       0.96      0.95      0.95       798
           5       0.98      0.98      0.98       798

    accuracy                           0.97      3989
   macro avg       0.97      0.97      0.97      3989
weighted avg       0.97      0.97      0.97      3989



In [232]:
param_dist = {
    'n_estimators': [300,400,500],
    'max_depth': [None,20,30],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4],
    'max_features': ['sqrt','log2']
}

rs= RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    n_jobs=-1,
    random_state=42
)

In [233]:
rs.fit(x_train, y_train)
best_rf = rs.best_estimator_
print("Best Parameters:", rs.best_params_)



KeyboardInterrupt



In [ ]:

y_pred = best_rf.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:

classes = np.unique(y)
y_test_bin = label_binarize(y_test, classes=classes)
y_score = best_rf.predict_proba(x_test)

roc_auc = roc_auc_score(y_test_bin, y_score, multi_class='ovr')

print("Multi-class ROC-AUC:", roc_auc)


In [ ]:
import pickle
pickle.dump(best_rf,open('model.pkl','wb'))

In [ ]:
df['Video Streaming Quality'].value_counts()

In [ ]:
best_rf.score(x_test, y_test)

In [ ]:
print(x.columns.tolist())

In [ ]:
pip install --upgrade scikit-learn

In [ ]:
print("Test inside notebook:")
print(best_rf.predict([[-65, 250, 10, 2, 1, 0]]))

In [ ]:
df['Signal Strength (dBm)'].value_counts()

In [ ]:
df['Download Speed (Mbps)'].value_counts()

In [ ]:
pip install xgboost

In [ ]:
from xgboost import XGBClassifier


In [ ]:
y_train = y_train - y_train.min()
y_test = y_test - y_test.min()


In [ ]:
y_train

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_train_enc = encoder.fit_transform(y_train)
y_test_enc = encoder.transform(y_test)

xgb_model.fit(x_train, y_train_enc)


In [ ]:
# Create model
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train model
xgb_model.fit(x_train, y_train)

In [ ]:
y_pred=xgb_model.predict(x_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [236]:
import pickle
pickle.dump(xgb_model,open('model.pkl','wb'))